# 05 · Export of the shared dataset

Writes the derived data released with the paper to `derivatives/shared_data/` and zips them for upload:
theta-filtered and broadband Laplacian-referenced hippocampal signals, normalised (MNI) contact coordinates,
the memorability estimates and the behavioural responses. The format is described in `shared_data_README.md`,
which is copied into the export as `README.md`. With these files, the behavioural, ERP and theta results can be
reproduced without the raw recordings (see `06_features_from_shared_data.ipynb`).

* One HDF5 file per participant, `sub-01.h5` … `sub-15.h5`. Study codes are replaced by these labels, and
  no dates, clinical information or native-space coordinates are included.
* The theta signal is the complex Morlet wavelet output at 3.5 and 7 Hz (`features.theta_tfr`), i.e. the
  same coefficients from which theta power, phase clustering and phase speed are computed in
  `03_feature_extraction.ipynb`. It is stored from −500 ms to 1000 ms after image onset, plus one sample, so
  that phase speed can be computed for every sample of the analysis window.
* The broadband LFP (−500 to 1000 ms) is included when `config.SHARE_BROADBAND_LFP` is True.
* `behaviour.csv` has one row per image presentation: participant label, image ID, presentation number and
  response (no timestamps).

Input: `derivatives/epochs_hippocampus_aligned.pickle` (notebook 02), the trial tables and the electrode position
files. Output: `derivatives/shared_data/` and `derivatives/shared_data.zip`.

In [ ]:
import pickle
import shutil

import h5py
import numpy as np
import pandas as pd

import config
import features
import preprocessing

out_dir = config.SHARED_DATA_DIR
out_dir.mkdir(parents=True, exist_ok=True)
N_KEEP = int(round((config.ANALYSIS_TMAX - config.EPOCH_TMIN) * config.SFREQ))  # 768 samples: -500 to 1000 ms
N_EXPORT = N_KEEP + 1  # one extra sample for the phase speed at the last stored sample

# Public participant labels (the mapping to study codes is not exported)
SUBJECT_LABELS = {patient_id: f"sub-{i:02d}" for i, patient_id in enumerate(config.PATIENTS, start=1)}

## sEEG data and contact coordinates

In [ ]:
with open(config.ALIGNED_EPOCHS_FILE, "rb") as f:
    aligned_epochs = pickle.load(f)

contact_rows = []
for patient_id, epochs in aligned_epochs.items():
    subject = SUBJECT_LABELS[patient_id]
    sfreq = epochs.info["sfreq"]
    assert sfreq == config.SFREQ, f"{patient_id}: unexpected sampling rate {sfreq}"
    assert np.isclose(epochs.times[0], config.EPOCH_TMIN), f"{patient_id}: unexpected epoch start"
    assert len(epochs.times) > N_EXPORT, f"{patient_id}: epochs shorter than the export window"
    contacts = list(epochs.ch_names)
    n_trials, n_contacts = len(epochs), len(contacts)
    print(f"{subject}: {n_trials} trials, {n_contacts} contacts")

    theta = np.empty((n_trials, n_contacts, len(features.THETA_FREQS), N_EXPORT), dtype=np.complex64)
    missing = np.zeros((n_trials, n_contacts), dtype=bool)
    lfp = np.empty((n_trials, n_contacts, N_KEEP), dtype=np.float32) if config.SHARE_BROADBAND_LFP else None
    for c, contact in enumerate(contacts):
        data = epochs.get_data(picks=[contact])  # (trials, 1, times)
        theta[:, c] = features.theta_tfr(data, sfreq)[..., :N_EXPORT]
        missing[:, c] = np.all(data[:, 0, :N_KEEP] == 0, axis=-1)  # contact-trials without data
        if lfp is not None:
            lfp[:, c] = data[:, 0, :N_KEEP]

    position_file = (config.SEEG_DATA_ROOT / patient_id / "anatomy" / "ElectrodeLocations"
                     / "Electrode_locs_3.xlsx")
    mni = preprocessing.contact_coordinates(position_file, contacts, space=config.MNI_COORDINATE_SPACE)
    image_id = pd.to_numeric(epochs.metadata["image_ID"]).to_numpy(np.int64)
    memorability = epochs.metadata["vit_memorability"].to_numpy(np.float64)

    with h5py.File(out_dir / f"{subject}.h5", "w") as h5:
        h5.attrs["subject"] = subject
        h5.attrs["sfreq"] = sfreq
        h5.attrs["description"] = ("Hippocampal sEEG, first presentation of each image, trials without interictal "
                                   "epileptiform activity. Shaft-wise Laplacian reference, polarity aligned across "
                                   "contacts. theta: complex Morlet wavelet coefficients (3.5 Hz, 3 cycles; "
                                   "7 Hz, 6 cycles). lfp: broadband signal.")
        h5.create_dataset("theta", data=theta, compression="gzip", compression_opts=4)
        h5.create_dataset("times", data=epochs.times[:N_EXPORT])
        h5.create_dataset("freqs", data=np.array(features.THETA_FREQS))
        h5.create_dataset("n_cycles", data=np.array(features.THETA_CYCLES))
        h5.create_dataset("missing", data=missing)
        h5.create_dataset("contacts/name", data=np.array(contacts, dtype=h5py.string_dtype()))
        h5.create_dataset("contacts/mni", data=mni)
        h5.create_dataset("trials/image_id", data=image_id)
        h5.create_dataset("trials/memorability", data=memorability)
        if lfp is not None:
            h5.create_dataset("lfp", data=lfp, compression="gzip", compression_opts=4)

    contact_rows += [{"subject": subject, "contact": name, "x_mni": xyz[0], "y_mni": xyz[1], "z_mni": xyz[2],
                      "n_trials_with_data": int((~missing[:, c]).sum())}
                     for c, (name, xyz) in enumerate(zip(contacts, mni))]

contacts_table = pd.DataFrame(contact_rows)
contacts_table.to_csv(out_dir / "contacts.csv", index=False)
print(f"{contacts_table['subject'].nunique()} participants, {len(contacts_table)} contacts")
contacts_table[["x_mni", "y_mni", "z_mni"]].describe()

## Behaviour: one row per image presentation

Same selection as `01_behaviour.ipynb`: image trials only, presentations numbered in the order of the trial table.

In [ ]:
memorability_scores = pd.read_csv(config.MEMORABILITY_FILE).rename(columns={"image": "image_ID"})

behaviour_frames = []
for patient_id in config.PATIENTS:
    trials = pd.read_pickle(config.SEEG_DATA_ROOT / patient_id / f"analysis_{config.TASK}" / "trials.pkl")
    df = trials[["image_ID", "last_response"]].merge(memorability_scores[["image_ID"]], on="image_ID")
    df["presentation"] = df.groupby("image_ID").cumcount() + 1
    behaviour_frames.append(pd.DataFrame({"subject": SUBJECT_LABELS[patient_id],
                                          "image_id": df["image_ID"].to_numpy(),
                                          "presentation": df["presentation"].to_numpy(),
                                          "response": df["last_response"].to_numpy()}))
    del trials

behaviour = pd.concat(behaviour_frames, ignore_index=True)
behaviour.to_csv(out_dir / "behaviour.csv", index=False)
print(f"{len(behaviour)} presentations ({(behaviour['presentation'] <= 3).sum()} with presentation 1-3)")
behaviour["response"].value_counts(dropna=False)

## Documentation and archive

In [ ]:
shutil.copy(config.MEMORABILITY_FILE, out_dir / "memorability_scores.csv")
shutil.copy(config.REPO_DIR / "shared_data_README.md", out_dir / "README.md")
archive = shutil.make_archive(str(config.DERIVATIVES_DIR / "shared_data"), "zip", root_dir=out_dir)
print(f"Wrote {archive}")